# 모델 비교 · 시각화 노트북

여러 모델의 예측을 **같은 test 샘플 위에 겹쳐 그려** 우승 모델과 다른 방법들을 비교합니다.
Cursor에서 커널을 `python3.12` 로 선택한 뒤 위에서부터 순서대로 실행하세요.

**사용법**: 아래 *사용자 입력* 셀에서 `TRACK / SEQ_LEN / PRED_LEN` 으로 셀을 고르고,
`MODELS` 에 비교할 모델명을 나열한 뒤 `SAMPLE_INDICES` 로 볼 샘플을 지정합니다.

- 지표 표는 학습 때 계산된 `comparison.csv` 를 그대로 읽어옵니다(재계산 없음, 빠름).
- 시각화는 **지정한 샘플 인덱스에 대해서만** 각 모델을 추론하므로 CPU로도 가볍습니다.
- `log10` 변환 공간에서 다뤄지며, 이벤트 임계값도 내부적으로 `log10(임계값)` 으로 비교됩니다.
- 발산한 체크포인트(예: `xpatch`, seq2016 → val_loss=inf)는 자동으로 감지해 표시하고 그래프에서 건너뜁니다.


In [ ]:
# ── 사용자 입력 (여기만 수정) ────────────────────────────────────────────
TRACK    = "uni_a"          # uni_a(PARTICLE) | uni_b(XRAY) | multi(둘 다)
SEQ_LEN  = 288              # 288 · 864 · 2016
PRED_LEN = 144              # 144 · 288
FOLD     = 0
STRATEGY = "direct"

# 비교할 모델들 (우승모델을 맨 앞에 두면 색이 고정돼 보기 편합니다)
MODELS = ["segrnn_thuml", "itransformer", "dlinear", "tsmixer"]

SAMPLE_INDICES = [0, 100, 500]   # 시각화할 test 샘플 인덱스
SHOW_PHYSICAL  = False           # True=물리단위(10**x, y로그축) · False=log10 공간
USE_GPU        = False           # 벤치마크가 GPU를 쓰는 중이면 False 권장(추론은 CPU로 충분)
GPU_INDEX      = "1"             # KASI 규칙: GPU 1 고정


In [ ]:
# ── 환경/경로 설정 (torch import 전에 GPU 지정) ──────────────────────────
import os, sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX if USE_GPU else ""

REPO = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "tslib").exists() and (p / "main.py").exists():
        REPO = p; break
assert REPO is not None, "repo 루트(tslib/, main.py)를 찾지 못했습니다."
sys.path.insert(0, str(REPO)); os.chdir(REPO)
print("REPO:", REPO)


In [ ]:
# ── 학습과 동일한 config 재구성 (config 는 모델과 무관, 셀당 1개) ─────────
import torch
from tslib.configs.config import exp_parser, config_postprocess
from tslib.benchmark import driver

cell = {"track": TRACK, "seq_len": SEQ_LEN, "pred_len": PRED_LEN,
        "fold": FOLD, "strategy": STRATEGY}
# --models 값은 config 재구성에만 필요하고 데이터/윈도우 설정과 무관하므로
# 대표로 MODELS[0] 하나만 넘겨 config 를 만든다 (모델별 로드는 뒤에서 개별 수행).
argv = driver.cell_argv(cell, epochs=1, models=[MODELS[0]])
config = config_postprocess(exp_parser().parse_args(argv))

run_name  = driver.run_name_for(TRACK, SEQ_LEN, PRED_LEN, FOLD, STRATEGY)
ckpt_dir  = REPO / "runs" / run_name / "ckpt"
print("run_name :", run_name)
print("임계값(물리):", config.event_threshold, "| transform:", config.transform)

# 체크포인트 존재 여부로 MODELS 필터링
avail, missing = [], []
for m in MODELS:
    (avail if (ckpt_dir / f"{m}.ckpt").exists() else missing).append(m)
if missing:
    print("⚠️ 체크포인트 없음(건너뜀):", missing)
print("비교 대상:", avail)
assert avail, "비교할 체크포인트가 하나도 없습니다."


In [ ]:
# ── 데이터 로드 (학습 때와 동일한 fold/윈도우) ───────────────────────────
from tslib.data.loader import DataModule

bundle  = DataModule(config).setup()
test_ds = bundle.test_loader.dataset
TARGETS   = list(bundle.target_cols)      # ['p_gt10'] 또는 ['p_gt10','xrs_long']
TARGET_CH = list(bundle.target_indices)
CAD_MIN   = config.cadence_min
print(f"input_size(C)={bundle.input_size} · targets={TARGETS} · target_ch={TARGET_CH}")
print(f"test 윈도우 수: {len(test_ds):,}  (seq_len={SEQ_LEN}, pred_len={PRED_LEN})")


In [ ]:
# ── 지표 비교표 (학습 때 계산된 comparison.csv 를 그대로 읽음) ────────────
import pandas as pd
comp_path = REPO / "runs" / run_name / "score" / "comparison.csv"
comp = pd.read_csv(comp_path).set_index("model")
# 관심 컬럼만: val_loss, rmse, 그리고 타깃별 hss/far
cols = ["best_val_loss", "rmse"]
cols += [c for c in comp.columns if c.startswith(("hss_", "far_", "pod_"))]
table = comp.loc[[m for m in avail if m in comp.index], cols].round(4)
print("지표 비교 (HSS 높을수록·FAR 낮을수록 좋음):")
table


In [ ]:
# ── 모델 빌드 + 체크포인트 로드 (test_only_neural 과 동일 절차) ───────────
from tslib.model import build_model
from tslib.exp.lightning_model import ForecastModule
from tslib.exp.metrics import MetricContext

dev = torch.device("cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu")
ctx = MetricContext(thresholds=config.event_threshold,
                    transform=config.transform, target_cols=TARGETS)

MODULES = {}
for m in avail:
    model  = build_model(m, config, bundle.input_size, bundle.target_indices, strategy=STRATEGY)
    module = ForecastModule(model, config, ctx, strategy=STRATEGY)
    state  = torch.load(ckpt_dir / f"{m}.ckpt", map_location="cpu")
    module.load_state_dict(state["state_dict"])
    MODULES[m] = module.eval().to(dev)
    print(f"loaded: {m}")


In [ ]:
# ── 지정 샘플만 각 모델로 추론 (가벼움) ──────────────────────────────────
import numpy as np

def fetch_batch(indices):
    xs, ys = [], []
    for i in indices:
        x, y = test_ds[i]
        xs.append(x); ys.append(y)
    return torch.stack(xs), torch.stack(ys)   # (B,seq,C), (B,pred,T)

valid_idx = [i for i in SAMPLE_INDICES if 0 <= i < len(test_ds)]
XB, YB = fetch_batch(valid_idx)
TRUE = YB.numpy()                              # (B,pred,T)

PREDS, DIVERGED = {}, []
with torch.no_grad():
    for m, mod in MODULES.items():
        p = mod(XB.to(dev)).cpu().numpy()      # (B,pred,T)
        if np.isnan(p).any() or np.isinf(p).any():
            DIVERGED.append(m)                 # 발산 체크포인트
        PREDS[m] = p
if DIVERGED:
    print("⚠️ 발산(NaN/Inf) → 그래프에서 제외:", DIVERGED)
print("추론 완료:", [m for m in PREDS if m not in DIVERGED], "| 샘플:", valid_idx)


In [ ]:
# ── 시각화 유틸 ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
%matplotlib inline

_CYCLE = ["crimson", "royalblue", "darkorange", "green",
          "purple", "brown", "teal", "magenta", "olive", "slategray"]
COLORS = {m: _CYCLE[i % len(_CYCLE)] for i, m in enumerate(avail)}

def _thr_stored(j):
    thr = float(config.event_threshold[j])
    return np.log10(thr) if config.transform == "log10" else thr
def _phys(v):
    return 10.0 ** v if (SHOW_PHYSICAL and config.transform == "log10") else v

def event_indices(j=0, n=30, scan=20000):
    \"\"\"타깃 j 에서 이벤트(>=임계값)가 실제 발생한 test 인덱스 (앞 scan개 스캔).\"\"\"
    thr = _thr_stored(j); out = []
    for i in range(min(scan, len(test_ds))):
        _, y = test_ds[i]
        if (y[:, j].numpy() >= thr).any():
            out.append(i)
            if len(out) >= n: break
    return out

def compare(idx):
    \"\"\"한 test 샘플에서 여러 모델 예측을 겹쳐 비교 (타깃별 서브플롯).\"\"\"
    b = valid_idx.index(idx)
    xh = test_ds[idx][0].numpy()               # (seq,C)
    T = len(TARGETS)
    fig, axes = plt.subplots(T, 1, figsize=(12, 3.6 * T), squeeze=False)
    t_hist = np.arange(-SEQ_LEN, 0) * CAD_MIN / 60.0
    t_fut  = np.arange(0, PRED_LEN) * CAD_MIN / 60.0
    unit = "phys" if SHOW_PHYSICAL else "log10"
    for j in range(T):
        ax = axes[j][0]; thr = _thr_stored(j)
        ax.plot(t_hist, _phys(xh[:, TARGET_CH[j]]), color="0.6", lw=1.1, label="history")
        ax.plot(t_fut, _phys(TRUE[b, :, j]), color="black", lw=2.2, label="TRUE", zorder=5)
        for m in avail:
            if m in DIVERGED: continue
            ax.plot(t_fut, _phys(PREDS[m][b, :, j]), color=COLORS[m],
                    lw=1.5, ls="--", alpha=0.85, label=m)
        ax.axvline(0, color="0.8", lw=1)
        ax.axhline(_phys(thr), color="green", ls=":", lw=1.3,
                   label=f"thr={config.event_threshold[j]:g}")
        if SHOW_PHYSICAL and config.transform == "log10":
            ax.set_yscale("log")
        ev = bool((TRUE[b, :, j] >= thr).any())
        ax.set_title(f"[{TRACK} seq{SEQ_LEN}/pred{PRED_LEN}] idx={idx} | "
                     f"{TARGETS[j]} | true event={'O' if ev else 'X'}")
        ax.set_xlabel("time (h),  0 = forecast start")
        ax.set_ylabel(f"{TARGETS[j]} ({unit})")
        ax.legend(loc="upper left", fontsize=8, ncol=2); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print("실제 이벤트 샘플 예시 (target 0):", event_indices(0, n=15))


In [ ]:
# ── 지정한 인덱스 비교 시각화 ────────────────────────────────────────────
for i in valid_idx:
    compare(i)


### 팁
- **이벤트 표본 비교**: `event_indices(0)` 가 준 인덱스를 위 입력 셀 `SAMPLE_INDICES` 에 넣고
  추론 셀부터 다시 실행하세요. (우주기상 이벤트는 희소해 임의 인덱스는 대부분 잔잔합니다.)
- **모델 추가/교체**: `MODELS` 리스트만 바꿔 로드 셀부터 재실행. 우승모델을 맨 앞에 두면 색(crimson)이 고정됩니다.
- **물리단위**: `SHOW_PHYSICAL=True` → y축 로그스케일 + `10**x`.
- **다른 셀 비교**: `TRACK/SEQ_LEN/PRED_LEN` 변경 후 config 셀부터 재실행.
- 검정 실선=실제(TRUE), 점선=각 모델 예측, 초록 점선=이벤트 임계값. 우승모델(segrnn_thuml)이 임계값 근처에서
  실제 곡선을 얼마나 잘 따라가는지, 다른 모델들의 과경보(임계 위로 튀는 예측)와 대비해 보세요.
